In [1]:
import pandas as pd
import ast
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

# --- 1. M4 HARDWARE SETUP ---
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"🚀 Engine booting up on: {device}")

# --- 2. THE DATA PIPELINE (PyTorch Dataset) ---

class DDXPlusDataset(Dataset):

    def __init__(self, csv_filepath):
        print("Loading massive CSV into memory...")
        # Load the CSV using pandas
        self.df = pd.read_csv(csv_filepath)

        print("🚨 PANDAS SEES THESE COLUMNS:", self.df.columns.tolist())
        
        # Build dictionaries to convert strings to numbers
        print("Mapping symptoms and diseases to numerical indices...")
        self.all_diseases = self.df['PATHOLOGY'].unique().tolist()
        self.disease_to_idx = {d: i for i, d in enumerate(self.all_diseases)}
        
        # Extract all unique symptoms to figure out the size of the X matrix
        all_symptoms = set()
        for evidences_str in self.df['EVIDENCES']:
            # ast.literal_eval converts the string "['E_7', 'E_24']" into an actual Python list ['E_7', 'E_24']
            symptoms_list = ast.literal_eval(evidences_str)
            all_symptoms.update(symptoms_list)
            
        self.all_symptoms = list(all_symptoms)
        self.symptom_to_idx = {s: i for i, s in enumerate(self.all_symptoms)}
        
        self.num_symptoms = len(self.all_symptoms)
        self.num_classes = len(self.all_diseases)
        print(f"Detected {self.num_symptoms} unique symptoms and {self.num_classes} diseases.")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Create an empty patient vector of zeros
        x_tensor = torch.zeros(self.num_symptoms, dtype=torch.float32)
        
        # Flip the zeros to ones for the symptoms this patient has
        patient_symptoms = ast.literal_eval(row['EVIDENCES'])
        for symptom in patient_symptoms:
            symptom_idx = self.symptom_to_idx[symptom]
            x_tensor[symptom_idx] = 1.0 
            
        # Get the target disease index
        y_idx = self.disease_to_idx[row['PATHOLOGY']]
        y_tensor = torch.tensor(y_idx, dtype=torch.long)
        
        return x_tensor, y_tensor

# --- 3. THE ARCHITECTURE (Softmax Model) ---
class ClinicalTriageModel(nn.Module):
    def __init__(self, input_size, num_classes):
        super(ClinicalTriageModel, self).__init__()
        # This is exactly equal to your NumPy code: Z = XW + b
        self.linear = nn.Linear(input_size, num_classes)

    def forward(self, x):
        return self.linear(x)

# --- 4. EXECUTION SCRIPT ---
if __name__ == "__main__":
    # Point this to your actual downloaded DDXPlus CSV!
    # For now, make sure you have 'release_validate_patients.csv' in your data folder
    CSV_PATH = "../data/raw/release_train_patients.csv" 
    
    # 1. Initialize Dataset and DataLoader
    dataset = DDXPlusDataset(CSV_PATH)
    # Batch size of 1024 keeps your 16GB RAM completely safe!
    dataloader = DataLoader(dataset, batch_size=1024, shuffle=True)
    
    # 2. Initialize Model and send to M4 chip
    model = ClinicalTriageModel(dataset.num_symptoms, dataset.num_classes).to(device)
    
    # 3. Setup Optimizer and Loss
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    
    # 4. The Training Loop!
    epochs = 20
    print("\n🔥 Starting Training Loop on M4 GPU...")
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        
        for batch_X, batch_y in dataloader:
            # Move the small batch to the GPU
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            
            optimizer.zero_grad()                 # Clear old gradients
            predictions = model(batch_X)          # Forward Pass
            loss = criterion(predictions, batch_y)# Calculate Error
            loss.backward()                       # Backward Pass (Matrix Calculus)
            optimizer.step()                      # Update Weights
            
            epoch_loss += loss.item()
            
        print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss/len(dataloader):.4f}")
        
    print("✅ Enterprise Training Complete!")

🚀 Engine booting up on: mps
Loading massive CSV into memory...
🚨 PANDAS SEES THESE COLUMNS: ['AGE', 'DIFFERENTIAL_DIAGNOSIS', 'SEX', 'PATHOLOGY', 'EVIDENCES', 'INITIAL_EVIDENCE']
Mapping symptoms and diseases to numerical indices...
Detected 515 unique symptoms and 49 diseases.

🔥 Starting Training Loop on M4 GPU...
Epoch 1/20 | Loss: 0.0681
Epoch 2/20 | Loss: 0.0096
Epoch 3/20 | Loss: 0.0085
Epoch 4/20 | Loss: 0.0081
Epoch 5/20 | Loss: 0.0079


KeyboardInterrupt: 